# Benchmark 2.0 — Perbandingan Arsitektur Deep Learning untuk Klasifikasi Jenis Sapi

**SapiKenal Research** | Politeknik Negeri Semarang

Notebook ini melakukan perbandingan **6 arsitektur CNN** menggunakan:
- **Stratified 5-Fold Cross-Validation** untuk evaluasi reliable pada dataset kecil (143 gambar)
- **2-Phase Training**: Feature Extraction (frozen backbone) → Fine-Tuning (unfreeze top 30%)
- **6 Arsitektur**: MobileNetV3Large, MobileNetV2, EfficientNetB0, EfficientNetV2S, ConvNeXtTiny, ResNet50
- **Evaluasi Komprehensif**: Per-class P/R/F1, Grad-CAM, McNemar's test, TFLite deployment

Dataset: [cattle-breeds-zrtzt v2](https://universe.roboflow.com/valll-rtclj/cattle-breeds-zrtzt/dataset/2) (Roboflow)

In [ ]:
# ========================================
# 1. Setup Lingkungan & Verifikasi GPU
# ========================================
!pip install -q roboflow

import os, gc, time, math, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path
from collections import Counter

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from scipy import stats as scipy_stats
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical, load_img, img_to_array

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 150
sns.set_style('whitegrid')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Terdeteksi: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("GPU tidak terdeteksi — disarankan aktifkan T4 di Runtime > Change runtime type.")
print(f"TensorFlow: {tf.__version__}")

## 2. Pemuatan & Eksplorasi Dataset

Dataset diunduh dari Roboflow, lalu **semua split (train/valid/test) digabung** menjadi satu pool
untuk digunakan dalam Stratified 5-Fold Cross-Validation.

In [ ]:
# ========================================
# 2. Download & Gabung Dataset
# ========================================
from roboflow import Roboflow

ROBOFLOW_API_KEY = "oSqbdWgbnJkiVfOWl18o"
WORKSPACE_NAME = "valll-rtclj"
PROJECT_NAME = "cattle-breeds-zrtzt"  # Sesuaikan jika berbeda
DATASET_VERSION = 2

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
dataset = project.version(DATASET_VERSION).download("folder")
DATASET_DIR = dataset.location

IMAGE_SIZE = (224, 224)

# Kumpulkan semua gambar dari semua split
all_paths = []
all_labels = []

for split_name in ['train', 'valid', 'test']:
    split_dir = os.path.join(DATASET_DIR, split_name)
    if not os.path.exists(split_dir):
        continue
    for class_name in sorted(os.listdir(split_dir)):
        class_dir = os.path.join(split_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for img_file in sorted(os.listdir(class_dir)):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                all_paths.append(os.path.join(class_dir, img_file))
                all_labels.append(class_name)

print(f"Total gambar: {len(all_paths)}")

class_names = sorted(set(all_labels))
NUM_CLASSES = len(class_names)
label_counts = Counter(all_labels)

print(f"Kelas ({NUM_CLASSES}): {class_names}")
for cls in class_names:
    print(f"  {cls}: {label_counts[cls]} gambar")

# Load semua gambar ke memori (143 gambar ~ 50MB, aman di Colab)
print("\nMemuat gambar ke memori...")
X_all = np.zeros((len(all_paths), *IMAGE_SIZE, 3), dtype=np.float32)
for i, path in enumerate(all_paths):
    X_all[i] = img_to_array(load_img(path, target_size=IMAGE_SIZE))

label_to_idx = {name: idx for idx, name in enumerate(class_names)}
y_all = np.array([label_to_idx[l] for l in all_labels])
y_all_cat = to_categorical(y_all, NUM_CLASSES)

print(f"Dataset loaded: X={X_all.shape}, y={y_all.shape}")

# Visualisasi distribusi
fig, ax = plt.subplots(figsize=(8, 4))
counts = [label_counts[c] for c in class_names]
bars = ax.bar(class_names, counts, color='#2b5c8f', alpha=0.85)
ax.set_title('Distribusi Dataset (Seluruh Split Digabung)', fontweight='bold')
ax.set_ylabel('Jumlah Gambar')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Konfigurasi Model & Fungsi Utilitas

### Arsitektur yang Dibandingkan
| Model | Alasan |
|---|---|
| MobileNetV3Large | Baseline SapiKenal |
| MobileNetV2 | Paling ringan |
| EfficientNetB0 | Compound scaling |
| EfficientNetV2S | Generasi terbaru EfficientNet |
| ConvNeXtTiny | CNN modern setara ViT |
| ResNet50 | Baseline klasik |

### Training Protocol
- **Phase 1**: Backbone frozen, lr=1e-3, max 30 epochs
- **Phase 2**: Top 30% backbone di-unfreeze, lr=1e-5, max 20 epochs

In [ ]:
# ========================================
# 3. Konfigurasi & Fungsi Utilitas
# ========================================
BATCH_SIZE = 16
PHASE1_EPOCHS = 30
PHASE2_EPOCHS = 20
PHASE1_LR = 1e-3
PHASE2_LR = 1e-5
N_FOLDS = 5
UNFREEZE_FRACTION = 0.3
RANDOM_STATE = 42

OUTPUT_DIR = Path("/content/benchmark2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = {
    "MobileNetV3Large": {
        "base_fn": tf.keras.applications.MobileNetV3Large,
        "preprocess_fn": tf.keras.applications.mobilenet_v3.preprocess_input,
    },
    "MobileNetV2": {
        "base_fn": tf.keras.applications.MobileNetV2,
        "preprocess_fn": tf.keras.applications.mobilenet_v2.preprocess_input,
    },
    "EfficientNetB0": {
        "base_fn": tf.keras.applications.EfficientNetB0,
        "preprocess_fn": tf.keras.applications.efficientnet.preprocess_input,
    },
    "EfficientNetV2S": {
        "base_fn": tf.keras.applications.EfficientNetV2S,
        "preprocess_fn": tf.keras.applications.efficientnet_v2.preprocess_input,
    },
    "ConvNeXtTiny": {
        "base_fn": tf.keras.applications.ConvNeXtTiny,
        "preprocess_fn": tf.keras.applications.convnext.preprocess_input,
    },
    "ResNet50": {
        "base_fn": tf.keras.applications.ResNet50,
        "preprocess_fn": tf.keras.applications.resnet50.preprocess_input,
    },
}


def build_model(model_name, num_classes, input_shape=(224, 224, 3)):
    config = MODEL_CONFIGS[model_name]
    base = config["base_fn"](weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D(name="gap")(x)
    x = Dense(256, activation="relu", name="dense_head")(x)
    x = Dropout(0.3, name="dropout_head")(x)
    out = Dense(num_classes, activation="softmax", name="output")(x)

    model = Model(inputs=base.input, outputs=out, name=model_name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE1_LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def unfreeze_top(model, fraction=0.3):
    # Kumpulkan layer backbone (sebelum GAP)
    base_layers = []
    for layer in model.layers:
        if layer.name == "gap":
            break
        base_layers.append(layer)

    # Freeze semua dulu
    for layer in base_layers:
        layer.trainable = False

    # Unfreeze top N% (skip BatchNorm agar stabil)
    n_unfreeze = max(1, int(len(base_layers) * fraction))
    for layer in base_layers[-n_unfreeze:]:
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True

    for layer in model.layers:
        if layer.name in ("dense_head", "dropout_head", "output"):
            layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE2_LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )


def get_callbacks():
    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=0),
    ]


def eval_model(model, X_val, y_val_idx, preprocess_fn):
    X_proc = preprocess_fn(X_val.copy())
    probs = model.predict(X_proc, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    return {
        "accuracy": accuracy_score(y_val_idx, y_pred),
        "precision": precision_score(y_val_idx, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_val_idx, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_val_idx, y_pred, average="macro", zero_division=0),
        "y_pred": y_pred,
        "per_class_p": precision_score(y_val_idx, y_pred, average=None, zero_division=0),
        "per_class_r": recall_score(y_val_idx, y_pred, average=None, zero_division=0),
        "per_class_f1": f1_score(y_val_idx, y_pred, average=None, zero_division=0),
    }

print(f"Model: {list(MODEL_CONFIGS.keys())}")
print(f"Config: {N_FOLDS}-Fold CV, Phase1={PHASE1_EPOCHS}ep, Phase2={PHASE2_EPOCHS}ep")

## 4. Training: 5-Fold CV × 6 Model × 2 Phase

Total: **60 training runs**. Estimasi waktu: ~2-3 jam di T4 GPU.

Progress disimpan setiap model selesai. Jika disconnect, set `RESUME = True` dan jalankan ulang cell ini.

In [ ]:
# ========================================
# 4. Training Loop
# ========================================
RESUME = False

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
checkpoint_path = OUTPUT_DIR / "checkpoint_results.pkl"

if RESUME and checkpoint_path.exists():
    with open(checkpoint_path, "rb") as f:
        all_results = pickle.load(f)
    completed = set(all_results.keys())
    print(f"Resumed. Sudah selesai: {completed}")
else:
    all_results = {}
    completed = set()

for model_name, config in MODEL_CONFIGS.items():
    if model_name in completed:
        print(f"SKIP {model_name} (sudah selesai)")
        continue

    print(f"\n{'='*70}")
    print(f"  MODEL: {model_name}")
    print(f"{'='*70}")

    mr = {
        "phase1": {"acc": [], "prec": [], "rec": [], "f1": [],
                   "y_trues": [], "y_preds": [],
                   "pc_p": [], "pc_r": [], "pc_f1": [], "hist": []},
        "phase2": {"acc": [], "prec": [], "rec": [], "f1": [],
                   "y_trues": [], "y_preds": [],
                   "pc_p": [], "pc_r": [], "pc_f1": [], "hist": []},
        "times": [], "best_acc": 0, "best_fold": 0,
    }

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all)):
        print(f"\n  Fold {fold+1}/{N_FOLDS}", end=" ")

        X_tr = X_all[tr_idx].copy()
        X_va = X_all[va_idx].copy()
        y_tr_c = y_all_cat[tr_idx]
        y_va_c = y_all_cat[va_idx]
        y_va_l = y_all[va_idx]

        aug_gen = ImageDataGenerator(
            preprocessing_function=config["preprocess_fn"],
            rotation_range=25, width_shift_range=0.1, height_shift_range=0.1,
            shear_range=0.1, zoom_range=0.2, horizontal_flip=True,
            brightness_range=[0.8, 1.2], fill_mode="nearest",
        )
        val_gen_maker = ImageDataGenerator(preprocessing_function=config["preprocess_fn"])

        steps_tr = math.ceil(len(X_tr) / BATCH_SIZE)
        steps_va = math.ceil(len(X_va) / BATCH_SIZE)

        model = build_model(model_name, NUM_CLASSES)
        t0 = time.time()

        # Phase 1: Feature Extraction (frozen backbone)
        h1 = model.fit(
            aug_gen.flow(X_tr, y_tr_c, batch_size=BATCH_SIZE, seed=RANDOM_STATE),
            steps_per_epoch=steps_tr, epochs=PHASE1_EPOCHS,
            validation_data=val_gen_maker.flow(X_va, y_va_c, batch_size=BATCH_SIZE, shuffle=False),
            validation_steps=steps_va, callbacks=get_callbacks(), verbose=0,
        )
        m1 = eval_model(model, X_va, y_va_l, config["preprocess_fn"])
        mr["phase1"]["acc"].append(m1["accuracy"])
        mr["phase1"]["prec"].append(m1["precision"])
        mr["phase1"]["rec"].append(m1["recall"])
        mr["phase1"]["f1"].append(m1["f1"])
        mr["phase1"]["y_trues"].append(y_va_l)
        mr["phase1"]["y_preds"].append(m1["y_pred"])
        mr["phase1"]["pc_p"].append(m1["per_class_p"])
        mr["phase1"]["pc_r"].append(m1["per_class_r"])
        mr["phase1"]["pc_f1"].append(m1["per_class_f1"])
        mr["phase1"]["hist"].append(h1.history)

        print(f"| P1: acc={m1['accuracy']:.3f} f1={m1['f1']:.3f}", end=" ")

        # Phase 2: Fine-Tuning (unfreeze top 30%)
        unfreeze_top(model, fraction=UNFREEZE_FRACTION)
        h2 = model.fit(
            aug_gen.flow(X_tr, y_tr_c, batch_size=BATCH_SIZE, seed=RANDOM_STATE + fold),
            steps_per_epoch=steps_tr, epochs=PHASE2_EPOCHS,
            validation_data=val_gen_maker.flow(X_va, y_va_c, batch_size=BATCH_SIZE, shuffle=False),
            validation_steps=steps_va, callbacks=get_callbacks(), verbose=0,
        )
        elapsed = time.time() - t0
        mr["times"].append(elapsed)

        m2 = eval_model(model, X_va, y_va_l, config["preprocess_fn"])
        mr["phase2"]["acc"].append(m2["accuracy"])
        mr["phase2"]["prec"].append(m2["precision"])
        mr["phase2"]["rec"].append(m2["recall"])
        mr["phase2"]["f1"].append(m2["f1"])
        mr["phase2"]["y_trues"].append(y_va_l)
        mr["phase2"]["y_preds"].append(m2["y_pred"])
        mr["phase2"]["pc_p"].append(m2["per_class_p"])
        mr["phase2"]["pc_r"].append(m2["per_class_r"])
        mr["phase2"]["pc_f1"].append(m2["per_class_f1"])
        mr["phase2"]["hist"].append(h2.history)

        print(f"| P2: acc={m2['accuracy']:.3f} f1={m2['f1']:.3f} | {elapsed:.0f}s")

        if m2["accuracy"] > mr["best_acc"]:
            mr["best_acc"] = m2["accuracy"]
            mr["best_fold"] = fold
            model.save(OUTPUT_DIR / f"best_{model_name}.keras")

        del model, X_tr, X_va
        tf.keras.backend.clear_session()
        gc.collect()

    all_results[model_name] = mr
    accs = mr["phase2"]["acc"]
    print(f"\n  >> {model_name} Phase 2: {np.mean(accs)*100:.2f}% +/- {np.std(accs)*100:.2f}%")

    # Checkpoint
    with open(checkpoint_path, "wb") as f:
        pickle.dump(all_results, f)

print("\n" + "="*70)
print("  SEMUA MODEL SELESAI!")
print("="*70)

## 5. Hasil & Analisis

In [ ]:
# ========================================
# 5a. Tabel Ringkasan Performa
# ========================================
summary_rows = []
for name in MODEL_CONFIGS:
    r = all_results[name]
    for phase_label, pk in [("Feature Extraction", "phase1"), ("Fine-Tuning", "phase2")]:
        p = r[pk]
        summary_rows.append({
            "Model": name, "Phase": phase_label,
            "Accuracy (%)": f"{np.mean(p['acc'])*100:.2f} +/- {np.std(p['acc'])*100:.2f}",
            "Precision": f"{np.mean(p['prec']):.4f} +/- {np.std(p['prec']):.4f}",
            "Recall": f"{np.mean(p['rec']):.4f} +/- {np.std(p['rec']):.4f}",
            "F1-Score": f"{np.mean(p['f1']):.4f} +/- {np.std(p['f1']):.4f}",
        })

summary_df = pd.DataFrame(summary_rows)
print("="*100)
print("  PERBANDINGAN PERFORMA — 5-FOLD STRATIFIED CV")
print("="*100)
display(summary_df)
summary_df.to_csv(OUTPUT_DIR / "benchmark2_results_summary.csv", index=False)

In [ ]:
# ========================================
# 5b. Metrik Per-Kelas (Phase 2: Fine-Tuning)
# ========================================
pc_rows = []
for name in MODEL_CONFIGS:
    p = all_results[name]["phase2"]
    mp = np.mean(p["pc_p"], axis=0)
    sp = np.std(p["pc_p"], axis=0)
    mr_ = np.mean(p["pc_r"], axis=0)
    sr = np.std(p["pc_r"], axis=0)
    mf = np.mean(p["pc_f1"], axis=0)
    sf = np.std(p["pc_f1"], axis=0)
    for ci, cn in enumerate(class_names):
        pc_rows.append({
            "Model": name, "Kelas": cn,
            "Precision": f"{mp[ci]:.4f} +/- {sp[ci]:.4f}",
            "Recall": f"{mr_[ci]:.4f} +/- {sr[ci]:.4f}",
            "F1-Score": f"{mf[ci]:.4f} +/- {sf[ci]:.4f}",
        })

pc_df = pd.DataFrame(pc_rows)
print("Metrik Per-Kelas (Phase 2, Mean +/- Std dari 5-Fold CV):\n")
display(pc_df)
pc_df.to_csv(OUTPUT_DIR / "benchmark2_per_class_metrics.csv", index=False)

In [ ]:
# ========================================
# 5c. Confusion Matrix Agregat (Phase 2)
# ========================================
fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes_flat = axes.flatten()

for idx, name in enumerate(MODEL_CONFIGS):
    p = all_results[name]["phase2"]
    yt = np.concatenate(p["y_trues"])
    yp = np.concatenate(p["y_preds"])
    cm = confusion_matrix(yt, yp, labels=range(NUM_CLASSES))
    acc = accuracy_score(yt, yp)
    f1 = f1_score(yt, yp, average="macro", zero_division=0)

    ax = axes_flat[idx]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{name}\n(Acc: {acc*100:.1f}% | F1: {f1:.3f})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")

plt.suptitle("Confusion Matrix Agregat — Phase 2 (Fine-Tuning)\n"
             f"5-Fold CV, {len(X_all)} Gambar Total",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices_phase2.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ========================================
# 5d. Variabilitas Fold & Perbandingan Phase
# ========================================
model_names_list = list(MODEL_CONFIGS.keys())
colors = ["#2b5c8f", "#3a7bbf", "#4e94d4", "#81b29a", "#a7c957", "#e07a5f"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot: Accuracy
p2_accs = [all_results[n]["phase2"]["acc"] for n in model_names_list]
bp = axes[0].boxplot(p2_accs, labels=[n.replace("Net", "\nNet") for n in model_names_list], patch_artist=True)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[0].set_title("Distribusi Accuracy per Fold\n(Phase 2: Fine-Tuning)", fontweight="bold")
axes[0].set_ylabel("Accuracy")

# Box plot: F1
p2_f1s = [all_results[n]["phase2"]["f1"] for n in model_names_list]
bp = axes[1].boxplot(p2_f1s, labels=[n.replace("Net", "\nNet") for n in model_names_list], patch_artist=True)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[1].set_title("Distribusi Macro F1 per Fold\n(Phase 2: Fine-Tuning)", fontweight="bold")
axes[1].set_ylabel("Macro F1")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fold_variability.png", dpi=200, bbox_inches="tight")
plt.show()

# Phase 1 vs Phase 2
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(model_names_list))
w = 0.35

p1m = [np.mean(all_results[n]["phase1"]["acc"])*100 for n in model_names_list]
p2m = [np.mean(all_results[n]["phase2"]["acc"])*100 for n in model_names_list]
p1s = [np.std(all_results[n]["phase1"]["acc"])*100 for n in model_names_list]
p2s = [np.std(all_results[n]["phase2"]["acc"])*100 for n in model_names_list]

b1 = ax.bar(x - w/2, p1m, w, yerr=p1s, label="Phase 1 (Frozen)", color="#2b5c8f", alpha=0.8, capsize=4)
b2 = ax.bar(x + w/2, p2m, w, yerr=p2s, label="Phase 2 (Fine-Tuned)", color="#e07a5f", alpha=0.8, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(model_names_list, rotation=25, ha="right")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Phase 1 vs Phase 2 (Mean +/- Std, 5-Fold CV)", fontweight="bold")
ax.legend()
ax.set_ylim(0, 110)

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 2, f"{h:.1f}%",
                ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "phase_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ========================================
# 6. Evaluasi Deployment (TFLite)
# ========================================
deployment = {}

for name in MODEL_CONFIGS:
    bp = OUTPUT_DIR / f"best_{name}.keras"
    if not bp.exists():
        print(f"  {name}: model file not found, skip")
        continue

    print(f"  Converting {name}...", end=" ")
    model = tf.keras.models.load_model(bp)
    total_params = model.count_params()

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_bytes = converter.convert()
    tflite_path = OUTPUT_DIR / f"{name}.tflite"
    with open(tflite_path, "wb") as f:
        f.write(tflite_bytes)

    tflite_mb = os.path.getsize(tflite_path) / (1024**2)
    keras_mb = os.path.getsize(bp) / (1024**2)

    interp = tf.lite.Interpreter(model_path=str(tflite_path))
    interp.allocate_tensors()
    inp_det = interp.get_input_details()
    dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
    lats = []
    for _ in range(110):
        t0 = time.time()
        interp.set_tensor(inp_det[0]["index"], dummy)
        interp.invoke()
        lats.append((time.time() - t0) * 1000)
    avg_lat = np.mean(lats[10:])  # skip warmup

    deployment[name] = {
        "params_m": total_params / 1e6,
        "keras_mb": keras_mb,
        "tflite_mb": tflite_mb,
        "latency_ms": avg_lat,
    }
    print(f"{total_params/1e6:.2f}M params | TFLite {tflite_mb:.1f}MB | {avg_lat:.1f}ms")

    del model, interp
    tf.keras.backend.clear_session()
    gc.collect()

# Tradeoff chart
dl = list(deployment.keys())
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

accs = [np.mean(all_results[n]["phase2"]["acc"])*100 for n in dl]
axes[0].bar(dl, accs, color="#2b5c8f", alpha=0.85)
axes[0].set_title("Test Accuracy (%)", fontweight="bold")
axes[0].set_ylim(0, 105)
for i, v in enumerate(accs):
    axes[0].text(i, v+1, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=9)

sizes = [deployment[n]["tflite_mb"] for n in dl]
axes[1].bar(dl, sizes, color="#e07a5f", alpha=0.85)
axes[1].set_title("Ukuran TFLite (MB)", fontweight="bold")
for i, v in enumerate(sizes):
    axes[1].text(i, v+0.5, f"{v:.1f}", ha="center", fontweight="bold", fontsize=9)

lats_vals = [deployment[n]["latency_ms"] for n in dl]
axes[2].bar(dl, lats_vals, color="#81b29a", alpha=0.85)
axes[2].set_title("Latensi TFLite CPU (ms)", fontweight="bold")
for i, v in enumerate(lats_vals):
    axes[2].text(i, v+0.5, f"{v:.1f}", ha="center", fontweight="bold", fontsize=9)

for ax in axes:
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("Tradeoff: Akurasi vs Efisiensi Deployment", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tradeoff_chart.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ========================================
# 7. Grad-CAM Visualization
# ========================================

def find_last_conv(model):
    for layer in reversed(model.layers):
        if len(layer.output.shape) == 4:
            return layer.name
    return None

def gradcam(model, img_batch, target_cls, layer_name):
    grad_model = Model(inputs=model.input,
                       outputs=[model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch)
        loss = preds[:, target_cls]
    grads = tape.gradient(loss, conv_out)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.reduce_sum(conv_out * weights[:, tf.newaxis, tf.newaxis, :], axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy()[0]

# Model terbaik berdasarkan mean accuracy
best_name = max(MODEL_CONFIGS.keys(),
                key=lambda n: np.mean(all_results[n]["phase2"]["acc"]))
print(f"Grad-CAM untuk model terbaik: {best_name}")
print(f"  Mean Acc: {np.mean(all_results[best_name]['phase2']['acc'])*100:.2f}%")

best_model = tf.keras.models.load_model(OUTPUT_DIR / f"best_{best_name}.keras")
target_layer = find_last_conv(best_model)
pfn = MODEL_CONFIGS[best_name]["preprocess_fn"]

fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(3*NUM_CLASSES, 6))

for ci in range(NUM_CLASSES):
    idx = np.where(y_all == ci)[0][0]
    raw = X_all[idx:idx+1].copy()
    proc = pfn(raw.copy())

    pred = best_model.predict(proc, verbose=0)
    pc = int(np.argmax(pred[0]))
    conf = pred[0][pc]

    hm = gradcam(best_model, proc, pc, target_layer)
    hm_resized = tf.image.resize(hm[..., np.newaxis], IMAGE_SIZE).numpy()[:, :, 0]

    axes[0, ci].imshow(raw[0].astype(np.uint8))
    axes[0, ci].set_title(class_names[ci], fontweight="bold", fontsize=10)
    axes[0, ci].axis("off")

    axes[1, ci].imshow(raw[0].astype(np.uint8))
    axes[1, ci].imshow(hm_resized, alpha=0.5, cmap="jet")
    label = class_names[pc]
    axes[1, ci].set_title(f"{label} ({conf:.0%})", fontsize=9)
    axes[1, ci].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Grad-CAM", fontsize=12, fontweight="bold")

plt.suptitle(f"Grad-CAM — {best_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "gradcam_examples.png", dpi=200, bbox_inches="tight")
plt.show()

del best_model
tf.keras.backend.clear_session()
gc.collect()

In [ ]:
# ========================================
# 8. Uji Signifikansi Statistik (McNemar's Test)
# ========================================
# Agregasi prediksi across all folds (tiap sampel muncul tepat 1x)
agg_preds = {}
y_true_global = None

for name in MODEL_CONFIGS:
    p = all_results[name]["phase2"]
    yt = np.concatenate(p["y_trues"])
    yp = np.concatenate(p["y_preds"])
    agg_preds[name] = yp
    if y_true_global is None:
        y_true_global = yt

# Pairwise McNemar
ml = list(MODEL_CONFIGS.keys())
mcn_rows = []

for i in range(len(ml)):
    for j in range(i+1, len(ml)):
        a, b = ml[i], ml[j]
        ac = agg_preds[a] == y_true_global
        bc = agg_preds[b] == y_true_global
        n01 = int(np.sum(ac & ~bc))  # A benar, B salah
        n10 = int(np.sum(~ac & bc))  # A salah, B benar

        if n01 + n10 > 0:
            chi2 = (abs(n01 - n10) - 1)**2 / (n01 + n10)
            pval = 1 - scipy_stats.chi2.cdf(chi2, 1)
        else:
            chi2, pval = 0.0, 1.0

        mcn_rows.append({
            "Model A": a, "Model B": b,
            "A>B": n01, "B>A": n10,
            "chi2": f"{chi2:.4f}", "p-value": f"{pval:.4f}",
            "Signifikan": "Ya" if pval < 0.05 else "Tidak",
        })

mcn_df = pd.DataFrame(mcn_rows)
print("McNemar's Test (Pairwise) — Phase 2:\n")
display(mcn_df)
mcn_df.to_csv(OUTPUT_DIR / "mcnemar_test_results.csv", index=False)

with open(OUTPUT_DIR / "statistical_significance.txt", "w") as f:
    f.write("McNemar's Test Results — Phase 2 (Fine-Tuning)\n")
    f.write("="*60 + "\n\n")
    f.write(mcn_df.to_string(index=False))
    f.write("\n\nH0: Kedua model memiliki error rate yang sama.\n")
    f.write("H1: Error rate berbeda secara signifikan (alpha=0.05).\n")

In [ ]:
# ========================================
# 9. Tabel Master & Ekspor
# ========================================
master_rows = []
for name in MODEL_CONFIGS:
    r = all_results[name]
    p2 = r["phase2"]
    dep = deployment.get(name, {})
    master_rows.append({
        "Model": name,
        "Accuracy (%)": f"{np.mean(p2['acc'])*100:.2f} +/- {np.std(p2['acc'])*100:.2f}",
        "Macro F1": f"{np.mean(p2['f1']):.4f} +/- {np.std(p2['f1']):.4f}",
        "Macro Precision": f"{np.mean(p2['prec']):.4f} +/- {np.std(p2['prec']):.4f}",
        "Macro Recall": f"{np.mean(p2['rec']):.4f} +/- {np.std(p2['rec']):.4f}",
        "Params (M)": f"{dep.get('params_m', 0):.2f}",
        "TFLite (MB)": f"{dep.get('tflite_mb', 0):.2f}",
        "Latency (ms)": f"{dep.get('latency_ms', 0):.2f}",
        "Train Time (s)": f"{np.mean(r['times']):.1f}",
    })

master_df = pd.DataFrame(master_rows)
print("="*100)
print("  TABEL MASTER — BENCHMARK 2.0 KLASIFIKASI JENIS SAPI")
print("  5-Fold Stratified CV | 2-Phase Training | 6 Arsitektur")
print("="*100)
display(master_df)
master_df.to_csv(OUTPUT_DIR / "benchmark2_master_table.csv", index=False)

# Detail per-fold
fold_rows = []
for name in MODEL_CONFIGS:
    r = all_results[name]
    for fi in range(N_FOLDS):
        fold_rows.append({
            "Model": name, "Fold": fi+1,
            "P1 Acc": f"{r['phase1']['acc'][fi]:.4f}",
            "P1 F1": f"{r['phase1']['f1'][fi]:.4f}",
            "P2 Acc": f"{r['phase2']['acc'][fi]:.4f}",
            "P2 F1": f"{r['phase2']['f1'][fi]:.4f}",
            "Time (s)": f"{r['times'][fi]:.1f}",
        })
fold_df = pd.DataFrame(fold_rows)
fold_df.to_csv(OUTPUT_DIR / "benchmark2_fold_details.csv", index=False)

print(f"\nSemua file disimpan di: {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.glob("*")):
    sz = f.stat().st_size / (1024**2)
    print(f"  {f.name}: {sz:.2f} MB")

## 10. Kesimpulan

Benchmark 2.0 telah mengevaluasi **6 arsitektur deep learning** untuk klasifikasi jenis sapi menggunakan:
- **5-Fold Stratified Cross-Validation** (evaluasi pada seluruh 143 gambar)
- **2-Phase Training Protocol** (Feature Extraction + Fine-Tuning)
- **Metrik komprehensif**: Accuracy, Precision, Recall, F1 (macro & per-class)
- **Analisis deployment**: TFLite size, latency CPU
- **Grad-CAM**: Interpretabilitas model (area citra yang diperhatikan)
- **McNemar's Test**: Uji signifikansi statistik antar model

### File Output
| File | Isi |
|---|---|
| `benchmark2_master_table.csv` | Tabel utama (accuracy, F1, params, TFLite, latency) |
| `benchmark2_results_summary.csv` | Phase 1 vs Phase 2 per model |
| `benchmark2_per_class_metrics.csv` | Precision/Recall/F1 per kelas per model |
| `benchmark2_fold_details.csv` | Detail tiap fold |
| `confusion_matrices_phase2.png` | Confusion matrix agregat 6 model |
| `fold_variability.png` | Box plot variabilitas antar fold |
| `phase_comparison.png` | Perbandingan Phase 1 vs Phase 2 |
| `tradeoff_chart.png` | Accuracy vs TFLite Size vs Latency |
| `gradcam_examples.png` | Visualisasi Grad-CAM model terbaik |
| `mcnemar_test_results.csv` | Uji signifikansi pairwise |

### Download Hasil
Jalankan cell berikut untuk mengunduh semua hasil sebagai ZIP.

In [ ]:
# ========================================
# Download semua hasil sebagai ZIP
# ========================================
import shutil
zip_path = shutil.make_archive("/content/benchmark2_outputs", "zip", "/content", "benchmark2_outputs")
print(f"ZIP: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Bukan di Colab — file tersedia di:", zip_path)